# MatchMaker — hyperparameter search (Optuna)

**`colab_run_matchmaker.ipynb` does not tune hyperparameters** — it runs a fixed grid of data × splits × seeds with the knobs you set once.

This notebook runs **Optuna trials**: each trial trains `main.py` on **one** processed dataset, **one** split, **one** seed, with a **capped `--max-epoch`** so search is affordable.

**Objective:** minimize **test MSE** from `results.csv` (same metric as the full pipeline). That uses the **test** split, which is standard for quick search but is optimistic if you reuse the same split for a final report — treat this as **exploration**, then lock hyperparameters and run the full matrix.

**Prerequisites:** same as the main Colab flow — repo root = `ens_492`, `data/` contains chem + GEX + processed TSVs + `splits/` for the chosen split/seed.


In [1]:
# 0) Install Optuna (Colab / local)
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
import optuna
print("optuna", optuna.__version__)


optuna 4.8.0


In [2]:
# 1) Configure search (edit here)
import os
from pathlib import Path

# Must match files under splits/<SPLIT_NAME>/seed_<SEED>/
SEED = 42
SPLIT = "lto"  # narrow search on one split first
DATASET_TSV = "data/processed/pancreatic_variance_filtered.tsv"  # or pancreatic_unfiltered.tsv

GPU_DEVICES = "0"
CLASSIFICATION_THRESHOLD = 0.0

# Cheaper search: small epoch cap + moderate early stopping (raise for final runs)
MAX_EPOCH_SEARCH = 120
EARLYSTOP_SEARCH = 25

N_TRIALS = 12  # increase if you have GPU time

HPO_OUT = Path("results/hpo_runs")
HPO_OUT.mkdir(parents=True, exist_ok=True)


In [3]:
from pathlib import Path

# 2) Objective: one main.py run per trial
import json
import os
import shutil
import subprocess
import sys
import time

import optuna
import pandas as pd

split_dir = Path("splits") / SPLIT / "seed_{}".format(SEED)
for name in ("train_inds.txt", "val_inds.txt", "test_inds.txt"):
    p = split_dir / name
    if not p.is_file():
        raise FileNotFoundError("Missing {} — run split generation first.".format(p))


def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-5, 3e-4, log=True)
    dropout = trial.suggest_float("dropout", 0.2, 0.6)
    input_dropout = trial.suggest_float("input_dropout", 0.1, 0.4)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    weight_mode = trial.suggest_categorical("weight_mode", ["uniform", "q3_upweight", "log"])
    norm = trial.suggest_categorical("norm", ["minmax", "tanh_norm"])

    outdir = HPO_OUT / "trial_{:04d}".format(trial.number)
    if outdir.exists():
        shutil.rmtree(outdir)
    outdir.mkdir(parents=True)

    cmd = [
        sys.executable,
        "-u",
        "main.py",
        "--comb-data-name",
        str(DATASET_TSV),
        "--label-column",
        "synergy_loewe",
        "--classification-label-column",
        "synergy_binary",
        "--classification-threshold",
        str(CLASSIFICATION_THRESHOLD),
        "--train-ind",
        str(split_dir / "train_inds.txt"),
        "--val-ind",
        str(split_dir / "val_inds.txt"),
        "--test-ind",
        str(split_dir / "test_inds.txt"),
        "--split-mode",
        "files",
        "--saved-model-name",
        "matchmaker.h5",
        "--outdir",
        str(outdir),
        "--gpu-devices",
        GPU_DEVICES,
        "--norm",
        norm,
        "--weight-mode",
        weight_mode,
        "--weight-alpha",
        "3.0",
        "--lr",
        str(lr),
        "--input-dropout",
        str(input_dropout),
        "--dropout",
        str(dropout),
        "--batch-size",
        str(batch_size),
        "--max-epoch",
        str(MAX_EPOCH_SEARCH),
        "--earlystop",
        str(EARLYSTOP_SEARCH),
        "--seed",
        str(SEED),
    ]

    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    t0 = time.monotonic()
    print("\n=== TRIAL {} ===\n{}".format(trial.number, " ".join(cmd)), flush=True)

    proc = subprocess.run(cmd, env=env)
    dt = time.monotonic() - t0
    print("trial {} finished in {:.1f}s rc={}".format(trial.number, dt, proc.returncode), flush=True)

    if proc.returncode != 0:
        return float('inf')

    res_path = outdir / "results.csv"
    if not res_path.is_file():
        return float("inf")
    df = pd.read_csv(res_path)
    mse = float(df["mse"].iloc[0])
    trial.set_user_attr("spearman", float(df["spearman"].iloc[0]))
    return mse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS)

best = study.best_params
print("Best MSE:", study.best_value)
print("Best params:", json.dumps(best, indent=2))

with open(HPO_OUT / "best_params.json", "w", encoding="utf-8") as f:
    json.dump({"best_value": study.best_value, "params": best, "split": SPLIT, "seed": SEED, "data": str(DATASET_TSV)}, f, indent=2)
print("Wrote", HPO_OUT / "best_params.json")




FileNotFoundError: Missing splits/lto/seed_42/train_inds.txt — run split generation first.

## Next steps

Copy the best knobs into **`colab_run_matchmaker.ipynb`** (LR, dropout, batch, `NORM`, `WEIGHT_MODE`) and run the **full** `run_experiments` grid with your production `MAX_EPOCH` / `EARLYSTOP`.

